# <span style="color: Skyblue;"> Scenario config UI </span>

In [6]:
import bcnexus.utils as utils

## 1: Define the Arguments & load _RunModel_ object

- Get the UI for user friendly inputs

In [7]:
scenarios_dd, storage_algo_dd, h_grouping_dd, n_clusters_dd, timeslices_label= utils.build_scenario_ui()

- Assign the values from dropdown to arguments

In [8]:
model_builder_args = dict(
    run_scenario=scenarios_dd.value ,
    storage_algorithm=storage_algo_dd.value,
    clustering_attributes=dict(
        hour_grouping=h_grouping_dd.value,
        n_clusters=n_clusters_dd.value
    )
)

# <span style="color: magenta;"> 2. Model Builder playground</span> 

 ## <span style="color: magenta;"> _*---------------------------------------------------------------*_
 </span> 

### 2A: Create an instance of the `BuildModel` Object, here named as `clewsBuild`

In [9]:
from bcnexus.clews.builder import BuildModel
clewsBuild=BuildModel(**model_builder_args)

____________________________________________________________________________________________________
     CLEWs Model Builder
____________________________________________________________________________________________________
******************
Scenario: Base_CNZ
******************
ℹ️  Using configuration file at: config/scenarios_bcnexus.yaml
ℹ️  Clustering attributes: {'hour_grouping': 12, 'n_clusters': 5}
ℹ️  Storage Algorithm: Kotzur
 └> Extracting class attributes e.g. directories, static values/ranges, constants etc.
ℹ️  bcnexus.attributes_parser | BCNexus CLEWs model is structured as SINGLE region model. Current region set to: REGION1.
ℹ️  bcnexus.attributes_parser | Fetching OTOOLE config from : models/model_Kotzur/otoole_config_Kotzur.yaml
 └> checking input CSVs...
 └> Copying missing CSV files : 'data/clews_data/csv_template' >> 'data/clews_data/clews_build_data/input_csvs'
  └─> Skipped (already exists): ReserveMarginTagFuel.csv
  └─> Skipped (already exists): DaySplit.csv

### 2B : Build SETs from Model Structure 
> formerly handled by clewsy tool

In [ ]:
from bcnexus.clews import sets_n_ratios
SetNames,NewSetItems,IARList,OARList=    sets_n_ratios.build('test_SETs')

In [ ]:
# clewsBuild.build_SETs_and_ratios()

- Copy the build SETs to `clews_build_data` directory.

In [ ]:
utils.copy_csv_files(src_folder='test_SETs', #clewsBuild.SETs_save_to, 
                     dest_folder=clewsBuild.clews_build_input_csv_dir, 
                     all_files=False)

### 2C : Update the Clews Builder Config to handle technology attributes, inclusion of new resource options and technology aggregations.
> Handles `STORAGE` types, `STORAGE_TECHNOLOGY` and non storage technologies named as `TECHNOLOGIES` 

In [ ]:
clewsBuild.get_clews_builder_attributes()

In [ ]:
clewsBuild.clewsb_config:dict=utils.load_config(clewsBuild.clews_builder_config_path)

In [ ]:
clewsBuild.update_temporal_profiles()
# clewsBuild.update_storage_case_SETs()

<span style="color: Yellow;"> 
 >> Now we have to update the associated SETs and Parameters to reflect the changes intended in clews_builder config
</span>

### 2D : Update SETs and Parameters

In [ ]:
clewsBuild.update_set_TECHNOLOGY()

In [ ]:
clewsBuild.update_set_STORAGE()

In [ ]:
clewsBuild.update_yearly_params() 

In [ ]:
clewsBuild.update_global_params()

In [ ]:
clewsBuild.trim_snapshot_data()

In [ ]:
clewsBuild.update_otoole_config()

In [ ]:
utils.copy_csv_files(src_folder=clewsBuild.clews_build_input_csv_dir,
                    dest_folder=clewsBuild.storage_case_input_csvs,
                    all_files=True)
# clewsBuild.update_storage_case_SETs()

 ## <span style="color: magenta;"> _*---------------------------------------------------------------*_
 </span> 

#  <span style="color: Yellow;">  3. Run Model </span> 

* <span style="color: grey;">From `clews.runner` module, load the `RunModel` Object as `clewsRun` instance </span>  

- `build=True` argument handles the workflow run mentioned in __Section 2__
- `update_temporal_profiles=True` collects profiling data 
  - from __'data/clews_data/clews_build_data/inputs_csv_8760'__ 
    - to __'data/clews_data/clews_build_data/Model_Kotzur/storage_case_input_csvs'__

## 1B : Load the instance of `RunModel` object
<span style="color: yellow;"> - Section 2 is to be used for Debugging and feature Developments</span> 

<span style="color: orange;">   - The sequential steps are included in Clews' Runner Object's method.</span> 

In [ ]:
clewsRun = RunModel(**clews_runner_args)

In [ ]:
clewsBuild.update_storage_case_temporal_schema()

In [ ]:
clewsRun = RunModel(**clews_runner_args)
clewsRun.process_scenario_data()

In [ ]:
data_path,model_path=clewsRun.get_model_run_files()

In [ ]:
clewsRun.write_LP_file(model_path,
                       data_path,
                       clewsRun.LP_file
                       )

In [ ]:
clewsRun.solve_model_gurobi(clewsRun.LP_file,
                            "log.txt")

In [ ]:
# clewsRun.run(update_temporal_profiles=False,
#              build=False,
#              include_livestock=False,
#              threads=32) # The thread depends on the hardware limitations of your machine. If you have 4 core CPU, use Thread <=4 )

## Load the Gurobi Model object

In [ ]:
# m=clewsRun.solved_model

In [ ]:
# m.printStats()          # Print basic stats: number of vars, constraints, etc.

In [ ]:
# m.getAttr('Obj')        # Get objective coefficients

In [ ]:
# m.getConstrs()          # List of all constraint objects

# <span style="color: yellow;"> 4 Check Results @ `results\clews\` </span>

In [ ]:
clewsRun.get_result_csvs(debug_mode=True)

# <span style="color: yellow;"> 5. Plot </span>

In [ ]:
import bcnexus.plots as plotter
plotter.main(args['scenario'],args['storage_algorithm'],timeslices=24)